# HECO validation test v.3 — diffusion-coefficient calibration

This notebook perturbs `sim_diffusion_coeff`, runs HECO with five worker processes, aggregates the 52 virtual-origin simulations into one convex-hull polygon for every simulation time, and evaluates the hull nearest the final SAR observation using the existing SRA, CI, Jaccard, and DICE definitions.

The full default design contains **150 diffusion coefficients × 52 virtual origins = 7,800 HECO runs**. Configuration generation and result writing are restart-safe. Simulation execution is disabled initially; inspect the experiment manifest and then set `RUN_SIMULATIONS = True`.


## 1. Imports, paths, and execution controls

The notebook is portable when placed in the `validation_test` directory. The explicit external-volume candidate is retained as a fallback for the current repository layout.


In [8]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from copy import deepcopy
from pathlib import Path
import json
import multiprocessing as mp
import os
import sys
import time
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore", category=FutureWarning)

validation_candidates = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path("/Volumes/LaCie/000 - Dottorato UNICT/seaquest_team/HECO/heco/validation_test"),
]
validation_dir = next(
    (
        path
        for path in validation_candidates
        if (path / "heco.py").is_file()
        and (path / "polygons_score.py").is_file()
    ),
    None,
)
if validation_dir is None:
    raise FileNotFoundError(
        "Could not locate validation_test (heco.py and polygons_score.py are required)."
    )

if str(validation_dir) not in sys.path:
    sys.path.insert(0, str(validation_dir))

SOURCE_POINTS = validation_dir / "virtual_points_oil_spill-test2/virtual_origin_oil_spill_points_2.geojson"
OBSERVED_FINAL_POLYGON = validation_dir / "results/qgis_comparison/30-AUG-2021-AbouSamra-and-Ai-2024-oilspill-SAR-detection.geojson"
FORCING_DATASET = validation_dir / "med-cmcc-cur-rean-h_1751886654126.nc"

experiment_dir = validation_dir / "calibration_D_test"
input_dir = experiment_dir / "input_yaml_files"
hulls_dir = experiment_dir / "convex_hulls"
final_hulls_dir = experiment_dir / "final_hulls"
tables_dir = experiment_dir / "tables"
for directory in (input_dir, hulls_dir, final_hulls_dir, tables_dir):
    directory.mkdir(parents=True, exist_ok=True)

N_WORKERS = 5
RUN_SIMULATIONS = True  # Change to True only after inspecting the manifest.
OVERWRITE_EXISTING = False
INTERPOLATED_VELOCITY = True
BASE_SEED = 20260721

for required_path in (SOURCE_POINTS, OBSERVED_FINAL_POLYGON, FORCING_DATASET):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print(f"Validation directory: {validation_dir}")
print(f"Experiment directory: {experiment_dir}")
print(f"Workers: {N_WORKERS}")


Validation directory: /Volumes/LaCie/000 - Dottorato UNICT/seaquest_team/HECO/heco/validation_test
Experiment directory: /Volumes/LaCie/000 - Dottorato UNICT/seaquest_team/HECO/heco/validation_test/calibration_D_test
Workers: 5


## 2. Experimental configuration

In [2]:
# Configuration drafted in the original calibration notebook.
base_config = {
    "input": {
        "dataset_file_name": str(FORCING_DATASET.resolve()),
        "lat0": 0.0,
        "lon0": 0.0,
        "sim_diffusion_coeff": 0.0,
        "sim_duration_h": 20.0,
        "sim_particles": 100,
        "sim_timedelta_s": 3600.0,
        "spill_release_duration_h": 1.0,
        "time0": "2021-08-29 16:36:07",
        "volume_spilled_m3": 100.0,
    }
}

N_DIFFUSION_TESTS = 150
DIFFUSION_MIN = 5.0
DIFFUSION_MAX = 80.0

# linspace includes both endpoints. The original i / num_tests expression did
# not include the requested upper limit.
diffusion_values = np.linspace(
    DIFFUSION_MIN,
    DIFFUSION_MAX,
    N_DIFFUSION_TESTS,
    endpoint=True,
)

with SOURCE_POINTS.open("r", encoding="utf-8") as handle:
    source_geojson = json.load(handle)

origin_records = []
for origin_id, feature in enumerate(source_geojson.get("features", [])):
    geometry = feature.get("geometry") or {}
    if geometry.get("type") != "Point":
        raise ValueError(f"Source feature {origin_id} is not a Point")
    lon, lat = geometry["coordinates"][:2]
    origin_records.append(
        {"origin_id": origin_id, "lat": float(lat), "lon": float(lon)}
    )

origins = pd.DataFrame(origin_records)
if origins.empty:
    raise ValueError("No virtual origins were found")

total_runs = len(diffusion_values) * len(origins)
print(f"Virtual origins: {len(origins)}")
print(f"Diffusion coefficients: {len(diffusion_values)}")
print(f"Total HECO simulations: {total_runs:,}")
print(f"Diffusion range: {diffusion_values[0]:.3f}–{diffusion_values[-1]:.3f}")
origins.head()


Virtual origins: 52
Diffusion coefficients: 150
Total HECO simulations: 7,800
Diffusion range: 5.000–80.000


,origin_id,lat,lon
0,0,35.354167,35.125001
1,1,35.395834,35.250001
2,2,35.354167,35.083334
3,3,35.354167,35.166668
4,4,35.687501,34.958334


## 3. Generate YAML configurations and manifest

Each diffusion coefficient is assigned a stable `diffusion_index`. Every coefficient is combined with all virtual origins. Deep copies prevent one configuration from leaking values into another.


In [3]:
manifest_rows = []

for diffusion_index, diffusion_coeff in enumerate(diffusion_values):
    for origin in origins.itertuples(index=False):
        config = deepcopy(base_config)
        config["input"]["lat0"] = float(origin.lat)
        config["input"]["lon0"] = float(origin.lon)
        config["input"]["sim_diffusion_coeff"] = float(diffusion_coeff)

        yaml_path = input_dir / (
            f"input_D{diffusion_index:03d}_point{int(origin.origin_id):03d}.yaml"
        )
        with yaml_path.open("w", encoding="utf-8") as handle:
            yaml.safe_dump(config, handle, sort_keys=False)

        manifest_rows.append(
            {
                "diffusion_index": diffusion_index,
                "sim_diffusion_coeff": float(diffusion_coeff),
                "origin_id": int(origin.origin_id),
                "lat": float(origin.lat),
                "lon": float(origin.lon),
                "yaml_path": str(yaml_path.resolve()),
            }
        )

manifest = pd.DataFrame(manifest_rows)
manifest_path = tables_dir / "diffusion_calibration_manifest.csv"
manifest.to_csv(manifest_path, index=False)

expected_rows = len(diffusion_values) * len(origins)
assert len(manifest) == expected_rows
assert manifest["yaml_path"].map(lambda value: Path(value).exists()).all()

print(f"Manifest rows: {len(manifest):,}")
print(f"Saved: {manifest_path}")
manifest.head()


Manifest rows: 7,800
Saved: /Volumes/LaCie/000 - Dottorato UNICT/seaquest_team/HECO/heco/validation_test/calibration_D_test/tables/diffusion_calibration_manifest.csv


,diffusion_index,sim_diffusion_coeff,origin_id,lat,lon,yaml_path
0,0,5.0,0,35.354167,35.125001,/Volumes/LaCie/000 - Dottorato UNICT/seaquest_...
1,0,5.0,1,35.395834,35.250001,/Volumes/LaCie/000 - Dottorato UNICT/seaquest_...
2,0,5.0,2,35.354167,35.083334,/Volumes/LaCie/000 - Dottorato UNICT/seaquest_...
3,0,5.0,3,35.354167,35.166668,/Volumes/LaCie/000 - Dottorato UNICT/seaquest_...
4,0,5.0,4,35.687501,34.958334,/Volumes/LaCie/000 - Dottorato UNICT/seaquest_...


## 4. Determine the scoring time

The observed GeoJSON timestamp is used when available. For each diffusion case, the nearest simulated convex-hull time is selected. With the supplied files, this preserves the previous comparison between the 30 August SAR polygon and the HECO hull around `2021-08-30 10:36:07`.


In [4]:
observed_layer = gpd.read_file(OBSERVED_FINAL_POLYGON)
if observed_layer.empty:
    raise ValueError("The observed SAR polygon layer is empty")

timestamp_columns = [
    column
    for column in observed_layer.columns
    if column.lower() in {"timestamp", "time", "datetime", "date"}
]
if timestamp_columns:
    observed_time = pd.to_datetime(
        observed_layer[timestamp_columns[0]].iloc[0],
        errors="raise",
    )
else:
    observed_time = pd.Timestamp("2021-08-30 10:36:07")
    warnings.warn(
        "No timestamp field was found in the observed polygon; using the previous scoring time."
    )

if observed_time.tzinfo is not None:
    observed_time = observed_time.tz_convert(None)

print(f"Observed polygon time: {observed_time}")
print(f"Observed geometry: {observed_layer.geometry.iloc[0].geom_type}")


Observed polygon time: 2021-08-30 10:58:28.229000
Observed geometry: Polygon


## 5. Worker: run one diffusion case, aggregate hulls, and score

One process handles one diffusion coefficient. It runs all origin configurations, merges their particle locations, builds a convex hull for each timestamp, writes only polygon products, selects the hull nearest the SAR timestamp, and applies `polygons_score.compute_metrics`. Particle GeoJSON files are intentionally not written.


In [5]:
def run_diffusion_case(job):
    # Imports inside the worker make the function independent of notebook state
    # after the process has forked.
    from contextlib import redirect_stderr, redirect_stdout
    from pathlib import Path
    import gc
    import io
    import os
    import sys
    import time
    import traceback

    import geopandas as gpd
    import numpy as np
    import pandas as pd
    from shapely.geometry import MultiPoint

    started = time.perf_counter()
    diffusion_index = int(job["diffusion_index"])
    diffusion_coeff = float(job["sim_diffusion_coeff"])

    result = {
        "diffusion_index": diffusion_index,
        "sim_diffusion_coeff": diffusion_coeff,
        "status": "error",
    }

    try:
        validation_path = Path(job["validation_dir"])
        os.chdir(validation_path)
        if str(validation_path) not in sys.path:
            sys.path.insert(0, str(validation_path))

        import heco
        from polygons_score import PROJECTED_CRS, compute_metrics

        output_parts = []
        for origin_id, yaml_path in job["configurations"]:
            seed = int(
                np.random.SeedSequence(
                    [int(job["base_seed"]), diffusion_index, int(origin_id)]
                ).generate_state(1, dtype=np.uint32)[0]
            )

            # HECO reports every release step to stdout. Suppress those messages
            # so parallel notebook output remains readable.
            with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
                output = heco.run(
                    str(yaml_path),
                    "uo",
                    "vo",
                    seed=seed,
                    interpolated=bool(job["interpolated"]),
                )

            if output is None or output.empty:
                raise ValueError(
                    f"HECO returned no particles for origin {origin_id}, D={diffusion_coeff}"
                )

            part = output[
                [
                    "release_step",
                    "sim_iteration",
                    "particle_id",
                    "time",
                    "lat",
                    "lon",
                ]
            ].copy()
            part["origin_id"] = int(origin_id)
            output_parts.append(part)

        particle_table = pd.concat(output_parts, ignore_index=True)
        particle_table["time"] = pd.to_datetime(
            particle_table["time"], errors="raise"
        )

        particle_gdf = gpd.GeoDataFrame(
            particle_table,
            geometry=gpd.points_from_xy(
                particle_table["lon"], particle_table["lat"]
            ),
            crs="EPSG:4326",
        )

        # Construct hulls in the same metric CRS used by polygons_score.py.
        # This avoids treating longitude and latitude as Cartesian distances.
        particle_projected = particle_gdf.to_crs(PROJECTED_CRS)

        hull_rows = []
        for timestamp, time_group in particle_projected.groupby("time", sort=True):
            geometry = MultiPoint(time_group.geometry.tolist()).convex_hull
            if geometry.is_empty or geometry.geom_type not in {
                "Polygon",
                "MultiPolygon",
            }:
                continue
            hull_rows.append(
                {
                    "time": pd.Timestamp(timestamp),
                    "sim_diffusion_coeff": diffusion_coeff,
                    "diffusion_index": diffusion_index,
                    "particle_count": int(len(time_group)),
                    "geometry": geometry,
                }
            )

        if not hull_rows:
            raise ValueError("No polygonal convex hulls were generated")

        hulls_projected = gpd.GeoDataFrame(
            hull_rows,
            geometry="geometry",
            crs=PROJECTED_CRS,
        )
        hulls = hulls_projected.to_crs("EPSG:4326")
        observed_time_local = pd.Timestamp(job["observed_time"])
        nearest_index = (hulls["time"] - observed_time_local).abs().idxmin()
        final_hull = hulls.loc[[nearest_index]].copy()
        score_time = pd.Timestamp(final_hull["time"].iloc[0])

        hulls_path = Path(job["hulls_dir"]) / (
            f"diffusion_{diffusion_index:03d}_all_times.geojson"
        )
        final_hull_path = Path(job["final_hulls_dir"]) / (
            f"diffusion_{diffusion_index:03d}_scored_hull.geojson"
        )

        for output_path in (hulls_path, final_hull_path):
            if output_path.exists():
                output_path.unlink()

        hulls_to_write = hulls.copy()
        hulls_to_write["time"] = hulls_to_write["time"].dt.strftime(
            "%Y-%m-%dT%H:%M:%S"
        )
        final_to_write = final_hull.copy()
        final_to_write["time"] = final_to_write["time"].dt.strftime(
            "%Y-%m-%dT%H:%M:%S"
        )
        hulls_to_write.to_file(hulls_path, driver="GeoJSON")
        final_to_write.to_file(final_hull_path, driver="GeoJSON")

        # Reuse the same projected-CRS formulas as the previous validation.
        metrics = compute_metrics(
            Path(job["observed_polygon"]),
            final_hull_path,
        )

        result.update(
            {
                "status": "ok",
                "score_time": score_time.isoformat(),
                "time_offset_minutes": abs(
                    (score_time - observed_time_local).total_seconds()
                )
                / 60.0,
                "SRA": metrics["SRA"],
                "CI": metrics["CI"],
                "Jaccard": metrics["J"],
                "DICE": metrics["DICE"],
                "area_observed_km2": metrics["area_O_km2"],
                "area_simulated_km2": metrics["area_S_km2"],
                "area_intersection_km2": metrics["area_inter_km2"],
                "area_union_km2": metrics["area_union_km2"],
                "centroid_distance_km": metrics["delta_x_km"],
                "observed_bbox_diagonal_km": metrics["L_box_km"],
                "particle_rows": int(len(particle_table)),
                "hull_count": int(len(hulls)),
                "hulls_path": str(hulls_path),
                "final_hull_path": str(final_hull_path),
                "error": "",
            }
        )
    except Exception as exc:
        result["error"] = f"{type(exc).__name__}: {exc}"
        result["traceback"] = traceback.format_exc(limit=8)
    finally:
        result["runtime_seconds"] = time.perf_counter() - started
        gc.collect()

    return result


## 6. Five-worker parallel execution

The checkpoint CSV is updated after every completed diffusion case. Successful cases are skipped when the cell is rerun unless `OVERWRITE_EXISTING = True`.

This notebook targets macOS/Linux and explicitly uses the `fork` multiprocessing context so a function defined in a notebook can be executed by worker processes. No NetCDF dataset is opened in the parent before the pool is created.


In [9]:
scores_path = tables_dir / "diffusion_calibration_scores.csv"

if scores_path.exists():
    checkpoint = pd.read_csv(scores_path)
else:
    checkpoint = pd.DataFrame()

completed_indices = set()
if not OVERWRITE_EXISTING and not checkpoint.empty:
    completed_indices = set(
        checkpoint.loc[checkpoint["status"] == "ok", "diffusion_index"]
        .astype(int)
        .tolist()
    )

jobs = []
for diffusion_index, case_rows in manifest.groupby("diffusion_index", sort=True):
    diffusion_index = int(diffusion_index)
    if diffusion_index in completed_indices:
        continue

    configurations = [
        (int(row.origin_id), str(row.yaml_path))
        for row in case_rows.sort_values("origin_id").itertuples(index=False)
    ]
    jobs.append(
        {
            "diffusion_index": diffusion_index,
            "sim_diffusion_coeff": float(
                case_rows["sim_diffusion_coeff"].iloc[0]
            ),
            "configurations": configurations,
            "validation_dir": str(validation_dir),
            "observed_polygon": str(OBSERVED_FINAL_POLYGON),
            "observed_time": pd.Timestamp(observed_time).isoformat(),
            "hulls_dir": str(hulls_dir),
            "final_hulls_dir": str(final_hulls_dir),
            "base_seed": BASE_SEED,
            "interpolated": INTERPOLATED_VELOCITY,
        }
    )

print(f"Already complete: {len(completed_indices)}")
print(f"Pending diffusion cases: {len(jobs)}")
print(f"Pending HECO runs: {len(jobs) * len(origins):,}")


def write_checkpoint(existing, new_records, destination):
    frames = [frame for frame in (existing, pd.DataFrame(new_records)) if not frame.empty]
    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not combined.empty:
        combined = (
            combined.sort_values(["diffusion_index", "status"])
            .drop_duplicates("diffusion_index", keep="last")
            .sort_values("diffusion_index")
            .reset_index(drop=True)
        )
        temporary_path = destination.with_suffix(".tmp")
        combined.to_csv(temporary_path, index=False)
        temporary_path.replace(destination)
    return combined


new_records = []
if not RUN_SIMULATIONS:
    print("Simulation execution is disabled. Set RUN_SIMULATIONS = True and rerun this cell.")
elif not jobs:
    print("All diffusion cases are already complete.")
else:
    if "fork" not in mp.get_all_start_methods():
        raise RuntimeError(
            "This notebook parallelization requires the 'fork' multiprocessing method."
        )

    context = mp.get_context("fork")
    started_all = time.perf_counter()

    with ProcessPoolExecutor(
        max_workers=N_WORKERS,
        mp_context=context,
    ) as executor:
        future_to_job = {
            executor.submit(run_diffusion_case, job): job for job in jobs
        }

        for completed_count, future in enumerate(as_completed(future_to_job), start=1):
            job = future_to_job[future]
            try:
                record = future.result()
            except Exception as exc:
                record = {
                    "diffusion_index": job["diffusion_index"],
                    "sim_diffusion_coeff": job["sim_diffusion_coeff"],
                    "status": "error",
                    "error": f"Worker failure: {type(exc).__name__}: {exc}",
                }

            new_records.append(record)
            checkpoint = write_checkpoint(checkpoint, new_records, scores_path)

            runtime_minutes = float(record.get("runtime_seconds", np.nan)) / 60.0
            print(
                f"[{completed_count:03d}/{len(jobs):03d}] "
                f"D={record['sim_diffusion_coeff']:.4f} "
                f"status={record['status']} "
                f"runtime={runtime_minutes:.1f} min"
            )
            if record["status"] != "ok":
                print(f"  {record.get('error', 'unknown error')}")

    elapsed_hours = (time.perf_counter() - started_all) / 3600.0
    print(f"Parallel calibration finished in {elapsed_hours:.2f} h")
    print(f"Checkpoint: {scores_path}")


Already complete: 0
Pending diffusion cases: 150
Pending HECO runs: 7,800
[001/150] D=5.5034 status=ok runtime=66.8 min
[002/150] D=6.0067 status=ok runtime=66.8 min
[003/150] D=6.5101 status=ok runtime=67.0 min
[004/150] D=5.0000 status=ok runtime=67.0 min
[005/150] D=7.0134 status=ok runtime=67.0 min
[006/150] D=7.5168 status=ok runtime=62.5 min
[007/150] D=8.0201 status=ok runtime=62.5 min
[008/150] D=9.5302 status=ok runtime=62.7 min
[009/150] D=8.5235 status=ok runtime=62.8 min
[010/150] D=9.0268 status=ok runtime=62.8 min
[011/150] D=10.0336 status=ok runtime=55.9 min
[012/150] D=10.5369 status=ok runtime=56.1 min
[013/150] D=11.5436 status=ok runtime=56.0 min
[014/150] D=11.0403 status=ok runtime=56.1 min
[015/150] D=12.0470 status=ok runtime=56.1 min
[016/150] D=12.5503 status=ok runtime=55.3 min
[017/150] D=13.0537 status=ok runtime=55.5 min
[018/150] D=14.0604 status=ok runtime=55.5 min
[019/150] D=13.5570 status=ok runtime=55.5 min
[020/150] D=14.5638 status=ok runtime=55.5 

## 7. Consolidate and rank calibration results

Higher values are preferred for SRA, Jaccard, and DICE; lower values are preferred for CI. The table reports the independently best diffusion coefficient for each metric rather than combining them into an undocumented composite score.


In [10]:
if not scores_path.exists():
    raise FileNotFoundError(
        f"No score table exists yet: {scores_path}. Run the parallel-execution cell first."
    )

scores = pd.read_csv(scores_path)
successful = scores.loc[scores["status"] == "ok"].copy()
failed = scores.loc[scores["status"] != "ok"].copy()

if successful.empty:
    raise ValueError("The checkpoint contains no successful diffusion cases")

numeric_columns = [
    "sim_diffusion_coeff",
    "SRA",
    "CI",
    "Jaccard",
    "DICE",
    "time_offset_minutes",
    "runtime_seconds",
]
for column in numeric_columns:
    successful[column] = pd.to_numeric(successful[column], errors="coerce")

best_rows = []
metric_directions = {
    "SRA": "max",
    "CI": "min",
    "Jaccard": "max",
    "DICE": "max",
}
for metric, direction in metric_directions.items():
    index = successful[metric].idxmax() if direction == "max" else successful[metric].idxmin()
    row = successful.loc[index]
    best_rows.append(
        {
            "metric": metric,
            "criterion": "higher is better" if direction == "max" else "lower is better",
            "best_value": row[metric],
            "sim_diffusion_coeff": row["sim_diffusion_coeff"],
            "diffusion_index": int(row["diffusion_index"]),
            "score_time": row["score_time"],
            "final_hull_path": row["final_hull_path"],
        }
    )

best_by_metric = pd.DataFrame(best_rows)
best_path = tables_dir / "diffusion_calibration_best_by_metric.csv"
best_by_metric.to_csv(best_path, index=False)

print(f"Successful diffusion cases: {len(successful)}")
print(f"Failed diffusion cases: {len(failed)}")
print(f"Best-by-metric table: {best_path}")
display(best_by_metric)

if not failed.empty:
    display(failed[["diffusion_index", "sim_diffusion_coeff", "error"]])


Successful diffusion cases: 150
Failed diffusion cases: 0
Best-by-metric table: /Volumes/LaCie/000 - Dottorato UNICT/seaquest_team/HECO/heco/validation_test/calibration_D_test/tables/diffusion_calibration_best_by_metric.csv


,metric,criterion,best_value,sim_diffusion_coeff,diffusion_index,score_time,final_hull_path
0,SRA,higher is better,0.962620,77.483221,144,2021-08-30T10:36:07,/Volumes/LaCie/000 - Dottorato UNICT/seaquest_...
1,CI,lower is better,0.069254,78.489933,146,2021-08-30T10:36:07,/Volumes/LaCie/000 - Dottorato UNICT/seaquest_...
2,Jaccard,higher is better,0.638580,10.033557,10,2021-08-30T10:36:07,/Volumes/LaCie/000 - Dottorato UNICT/seaquest_...
3,DICE,higher is better,0.779431,10.033557,10,2021-08-30T10:36:07,/Volumes/LaCie/000 - Dottorato UNICT/seaquest_...


## 8. Select the calibrated diffusion coefficient

A single coefficient is selected with an equal-weight, rank-based consensus of the three non-redundant criteria:

- maximize SRA, representing coverage of the observed SAR slick;
- minimize CI, representing centroid displacement;
- maximize Jaccard, representing overlap while penalizing both missed and excess simulated area.

DICE is reported for the selected case but is excluded from the consensus calculation because it is a monotonic transformation of Jaccard and therefore produces the same ordering. Including both would count the same overlap information twice. The candidate with the lowest mean rank is selected; ties are resolved by higher Jaccard, lower CI, higher SRA, and finally lower `diffusion_index`.


In [11]:
selection_columns = [
    "diffusion_index",
    "sim_diffusion_coeff",
    "SRA",
    "CI",
    "Jaccard",
    "DICE",
    "score_time",
    "time_offset_minutes",
    "final_hull_path",
]

selection_candidates = successful[selection_columns].dropna(
    subset=["SRA", "CI", "Jaccard", "DICE"]
).copy()
if selection_candidates.empty:
    raise ValueError("No complete metric rows are available for final selection")

# Rank 1 is best for every criterion. Average ranks make the decision independent
# of the different numerical ranges used by SRA, CI, and Jaccard.
selection_candidates["rank_SRA"] = selection_candidates["SRA"].rank(
    method="average", ascending=False
)
selection_candidates["rank_CI"] = selection_candidates["CI"].rank(
    method="average", ascending=True
)
selection_candidates["rank_Jaccard"] = selection_candidates["Jaccard"].rank(
    method="average", ascending=False
)
selection_candidates["consensus_mean_rank"] = selection_candidates[
    ["rank_SRA", "rank_CI", "rank_Jaccard"]
].mean(axis=1)

consensus_ranking = selection_candidates.sort_values(
    by=[
        "consensus_mean_rank",
        "Jaccard",
        "CI",
        "SRA",
        "diffusion_index",
    ],
    ascending=[True, False, True, False, True],
).reset_index(drop=True)
consensus_ranking.insert(0, "selection_rank", np.arange(1, len(consensus_ranking) + 1))

selected = consensus_ranking.iloc[0]
CALIBRATED_SIM_DIFFUSION_COEFF = float(selected["sim_diffusion_coeff"])

selected_summary = pd.DataFrame(
    [
        {
            "parameter": "sim_diffusion_coeff",
            "selected_value": CALIBRATED_SIM_DIFFUSION_COEFF,
            "diffusion_index": int(selected["diffusion_index"]),
            "selection_method": "lowest equal-weight mean rank across SRA, CI, and Jaccard",
            "consensus_mean_rank": float(selected["consensus_mean_rank"]),
            "SRA": float(selected["SRA"]),
            "CI": float(selected["CI"]),
            "Jaccard": float(selected["Jaccard"]),
            "DICE": float(selected["DICE"]),
            "score_time": selected["score_time"],
            "time_offset_minutes": float(selected["time_offset_minutes"]),
            "observed_polygon": str(OBSERVED_FINAL_POLYGON),
            "selected_hull": selected["final_hull_path"],
        }
    ]
)

ranking_path = tables_dir / "diffusion_calibration_consensus_ranking.csv"
selection_path = tables_dir / "diffusion_calibration_selected.csv"
selection_yaml_path = tables_dir / "diffusion_calibration_selected.yaml"

consensus_ranking.to_csv(ranking_path, index=False)
selected_summary.to_csv(selection_path, index=False)

selection_yaml = {
    "calibration": {
        "parameter": "sim_diffusion_coeff",
        "selected_value": CALIBRATED_SIM_DIFFUSION_COEFF,
        "diffusion_index": int(selected["diffusion_index"]),
        "selection_method": (
            "lowest equal-weight mean rank across SRA (maximize), "
            "CI (minimize), and Jaccard (maximize)"
        ),
        "dice_usage": (
            "reported but excluded from selection because it is a monotonic "
            "transformation of Jaccard"
        ),
        "metrics": {
            "SRA": float(selected["SRA"]),
            "CI": float(selected["CI"]),
            "Jaccard": float(selected["Jaccard"]),
            "DICE": float(selected["DICE"]),
        },
        "score_time": str(selected["score_time"]),
        "observed_polygon": str(OBSERVED_FINAL_POLYGON),
        "selected_hull": str(selected["final_hull_path"]),
        "base_seed": int(BASE_SEED),
        "virtual_origins": int(len(origins)),
        "tested_coefficients": int(len(diffusion_values)),
    }
}
with selection_yaml_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(selection_yaml, handle, sort_keys=False)

print("Selected HECO calibration value")
print(f"sim_diffusion_coeff = {CALIBRATED_SIM_DIFFUSION_COEFF:.6g}")
print(f"Consensus ranking: {ranking_path}")
print(f"Selection record: {selection_path}")
print(f"Selection YAML: {selection_yaml_path}")

display(selected_summary.T.rename(columns={0: "selected case"}))
display(
    consensus_ranking[
        [
            "selection_rank",
            "sim_diffusion_coeff",
            "consensus_mean_rank",
            "SRA",
            "CI",
            "Jaccard",
            "DICE",
        ]
    ].head(10)
)


Selected HECO calibration value
sim_diffusion_coeff = 41.2416
Consensus ranking: /Volumes/LaCie/000 - Dottorato UNICT/seaquest_team/HECO/heco/validation_test/calibration_D_test/tables/diffusion_calibration_consensus_ranking.csv
Selection record: /Volumes/LaCie/000 - Dottorato UNICT/seaquest_team/HECO/heco/validation_test/calibration_D_test/tables/diffusion_calibration_selected.csv
Selection YAML: /Volumes/LaCie/000 - Dottorato UNICT/seaquest_team/HECO/heco/validation_test/calibration_D_test/tables/diffusion_calibration_selected.yaml


,selected case
parameter,sim_diffusion_coeff
selected_value,41.241611
diffusion_index,72
selection_method,"lowest equal-weight mean rank across SRA, CI, ..."
consensus_mean_rank,34.0
SRA,0.915872
CI,0.078876
Jaccard,0.636382
DICE,0.777792
score_time,2021-08-30T10:36:07


,selection_rank,sim_diffusion_coeff,consensus_mean_rank,SRA,CI,Jaccard,DICE
0,1,41.241611,34.000000,0.915872,0.078876,0.636382,0.777792
1,2,72.449664,36.000000,0.953131,0.078740,0.611494,0.758916
2,3,63.389262,37.000000,0.954888,0.078140,0.608901,0.756915
3,4,32.181208,39.333333,0.899424,0.079811,0.637552,0.778665
4,5,35.201342,39.333333,0.911873,0.079029,0.632910,0.775193
5,6,69.429530,43.000000,0.954474,0.075222,0.597887,0.748347
6,7,66.409396,43.666667,0.949345,0.075237,0.600983,0.750767
7,8,62.885906,44.000000,0.957533,0.083236,0.611936,0.759256
8,9,25.134228,45.000000,0.881120,0.080382,0.637709,0.778782
9,10,67.919463,45.000000,0.956945,0.080088,0.600885,0.750691


## Output structure

- `calibration_D_test/input_yaml_files/`: one HECO YAML per diffusion/origin combination.
- `calibration_D_test/convex_hulls/`: time-resolved aggregated convex hulls, one file per diffusion coefficient.
- `calibration_D_test/final_hulls/`: the hull nearest the SAR timestamp, one file per coefficient.
- `calibration_D_test/tables/diffusion_calibration_manifest.csv`: complete experiment manifest.
- `calibration_D_test/tables/diffusion_calibration_scores.csv`: restart-safe metrics checkpoint.
- `calibration_D_test/tables/diffusion_calibration_best_by_metric.csv`: independently optimal coefficients for SRA, CI, Jaccard, and DICE.
- `calibration_D_test/tables/diffusion_calibration_consensus_ranking.csv`: complete final-selection ranking.
- `calibration_D_test/tables/diffusion_calibration_selected.csv`: selected coefficient and associated SAR benchmark scores.
- `calibration_D_test/tables/diffusion_calibration_selected.yaml`: machine-readable calibrated value and selection provenance.
